In [1]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro
from scipy import stats
import math
from matplotlib.ticker import MaxNLocator

# Show all rows and all columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Show full contents of each cell (no truncation)
pd.set_option("display.max_colwidth", None)

# Make the dataframe print across the full console width
pd.set_option("display.width", 0)

In [2]:
# NORMALISATION CHECKS

df = pd.read_csv('../data/Jacob Spectrum Report for 20240628_Rats_no_header.csv')

In [3]:
df = df.rename(columns = {'Variable modifications identified by spectrum' : 'pep_mods'})
df = df.rename(columns = {'Peptide start index' : 'pep_start'})
df = df.rename(columns = {'Peptide stop index' : 'pep_end'})
df = df.rename(columns = {'MS/MS sample name' : 'sample'})
df["pep_mods"] = df["pep_mods"].str.replace(r"Oxidation|Deamidated", "", regex=True)

In [4]:
# lds = list of datasets []
# m = specific modification 'oxP'
# l = loci range ( , )

def findtotalptms(lds, p, *args, **kwargs):    
    PTMdf = pd.DataFrame(columns = ['Sample', 'Prot', '50aa_reg_start', '50aa_reg_end', '50aa_reg_range', 'oxP', 'oxM', 'oxK', 'dNQ', 'SurrOxP', 'SurrOxM', 'SurrOxK', 'SurrDNQ'])
    for ds in lds:  
        ldslabels = ds['sample'].unique()
        print('ldslabels', ldslabels)
        ds["pep_mods"] = ds["pep_mods"].fillna("")
        ds["pep_mods"] = ds["pep_mods"].astype(str)
        for d in ldslabels:
            dataset = ds[ds['sample'] == d]
            prots = dataset['Alternate IDs'].unique()
            for prot in p:
                data = dataset[dataset['Alternate IDs'] == prot]
                maxaapos = data['pep_end'].max()
                max50aagap = math.ceil(maxaapos / 50) * 50
                increments = [(i, i + 50) for i in range(0, max50aagap, 50)]
                for r in increments:
                    datasubset = data[(data['pep_start'].between(r[0],r[1])) | (data['pep_end'].between(r[0],r[1]))]
                    surrdata = data[data['pep_start'].between(r[0] - 50,r[1] + 50) | data['pep_end'].between(r[0] - 50,r[1] + 50)]  
                    oxP = datasubset['pep_mods'].str.count('p').sum()
                    oxK = datasubset['pep_mods'].str.count('k').sum()
                    oxM = datasubset['pep_mods'].str.count('m').sum()
                    dNQ = (datasubset['pep_mods'].str.count('n').sum()) + (datasubset['pep_mods'].str.count('q').sum())

                    oxPsurr = surrdata['pep_mods'].str.count('p').sum()
                    oxKsurr = surrdata['pep_mods'].str.count('k').sum()
                    oxMsurr = surrdata['pep_mods'].str.count('m').sum()
                    dNQsurr = (surrdata['pep_mods'].str.count('n').sum()) + (surrdata['pep_mods'].str.count('q').sum())

                    PTMdf.loc[len(PTMdf)] = [data['sample'].iloc[0], prot, r[0], r[1], f'{r[0]} - {r[1]}', oxP, oxM, oxK, dNQ, oxPsurr, oxMsurr, oxKsurr, dNQsurr]

    return PTMdf


In [10]:
femproteinList = ['Apoe', 'Col1a1', 'Col1a2', 'Col2a1', 'Comp', 'Spp1', 'Serpinh1', 'F2']
maleproteinList = ['Apoe', 'Col1a1', 'Col1a2', 'Col2a1', 'Comp', 'Spp1', 'Serpinh1', 'F2']

juvquantmods = findtotalptms([juvdf], femproteinList)
adultquantmods = findtotalptms([adultdf], maleproteinList) 

ldslabels ['FZ1_F (ms-status_RAT1)' 'FZ2_F (ms-status_RAT2)'
 'FZ3_F (ms-status_RAT3)' 'FZ4_M (ms-status_RAT4)'
 'FZ5_M (ms-status_RAT5)' 'FZ6_M (ms-status_RAT6)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


ldslabels ['JA1_M (ms-status_RAT7)' 'JA2_F (ms-status_RAT8)'
 'JA3_M (ms-status_RAT9)' 'JA4_M (ms-status_RAT10)'
 'JA5_F (ms-status_RAT11)' 'JA6_F (ms-status_RAT12)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


In [11]:
proteinList = ['Apoe', 'Col1a1', 'Col1a2', 'Col2a1', 'Comp', 'Spp1', 'Serpinh1', 'F2']

MaleData = pd.concat([juvdf_m, adultdf_m])


malequantmods = findtotalptms([juvdf], proteinList)
femalequantmods = findtotalptms([adultdf], proteinList) 

ldslabels ['FZ1_F (ms-status_RAT1)' 'FZ2_F (ms-status_RAT2)'
 'FZ3_F (ms-status_RAT3)' 'FZ4_M (ms-status_RAT4)'
 'FZ5_M (ms-status_RAT5)' 'FZ6_M (ms-status_RAT6)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


ldslabels ['JA1_M (ms-status_RAT7)' 'JA2_F (ms-status_RAT8)'
 'JA3_M (ms-status_RAT9)' 'JA4_M (ms-status_RAT10)'
 'JA5_F (ms-status_RAT11)' 'JA6_F (ms-status_RAT12)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


In [12]:
adultmale = findtotalptms([adultdf_m], proteinList)
juvmale = findtotalptms([juvdf_m], proteinList)

adultfemale = findtotalptms([adultdf_f], proteinList)
juvfemale = findtotalptms([juvdf_f], proteinList)



ldslabels ['JA1_M (ms-status_RAT7)' 'JA3_M (ms-status_RAT9)'
 'JA4_M (ms-status_RAT10)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


ldslabels ['FZ3_F (ms-status_RAT3)' 'FZ4_M (ms-status_RAT4)'
 'FZ5_M (ms-status_RAT5)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


ldslabels ['JA2_F (ms-status_RAT8)' 'JA5_F (ms-status_RAT11)'
 'JA6_F (ms-status_RAT12)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


ldslabels ['FZ1_F (ms-status_RAT1)' 'FZ2_F (ms-status_RAT2)'
 'FZ6_M (ms-status_RAT6)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


In [13]:
def pivotmods(data):
    d = data
    d = d.pivot_table(
        index=["Prot","50aa_reg_range"],      # rows
        columns="Sample",                     # columns by sample
        values=["oxP","oxM","oxK","dNQ"],     # values
        fill_value=0                           # fill missing counts with 0
    )
    # d.drop(['SurrOxP', 'SurrOxM', 'SurrOxK', 'SurrDNQ'])

    # Optional: flatten multi-level columns
    d.columns = [f"{mod}_{sample}" for mod, sample in d.columns]
    d.reset_index(inplace=True)

    return d


In [14]:
malejuvmodspiv = pivotmods(juvmale)
maleadultmodspiv = pivotmods(adultmale)

femalejuvmodspiv = pivotmods(juvfemale)
femaleadultmodspiv = pivotmods(adultfemale)

In [15]:
femaleadultmodspiv.columns

Index(['Prot', '50aa_reg_range', 'dNQ_JA2_F (ms-status_RAT8)',
       'dNQ_JA5_F (ms-status_RAT11)', 'dNQ_JA6_F (ms-status_RAT12)',
       'oxK_JA2_F (ms-status_RAT8)', 'oxK_JA5_F (ms-status_RAT11)',
       'oxK_JA6_F (ms-status_RAT12)', 'oxM_JA2_F (ms-status_RAT8)',
       'oxM_JA5_F (ms-status_RAT11)', 'oxM_JA6_F (ms-status_RAT12)',
       'oxP_JA2_F (ms-status_RAT8)', 'oxP_JA5_F (ms-status_RAT11)',
       'oxP_JA6_F (ms-status_RAT12)'],
      dtype='object')

In [16]:
dNQCols = ['dNQ_JA2_F (ms-status_RAT8)', 'dNQ_JA5_F (ms-status_RAT11)', 'dNQ_JA6_F (ms-status_RAT12)']
oxKCols = ['oxK_JA2_F (ms-status_RAT8)', 'oxK_JA5_F (ms-status_RAT11)', 'oxK_JA6_F (ms-status_RAT12)']
oxMCols = ['oxM_JA2_F (ms-status_RAT8)', 'oxM_JA5_F (ms-status_RAT11)', 'oxM_JA6_F (ms-status_RAT12)']
oxPCols = ['oxP_JA2_F (ms-status_RAT8)', 'oxP_JA5_F (ms-status_RAT11)', 'oxP_JA6_F (ms-status_RAT12)']

for group, cols in {
    'AVG. dNQ': dNQCols,
    'AVG. oxK': oxKCols,
    'AVG. oxM': oxMCols,
    'AVG. oxP': oxPCols
}.items():
    femaleadultmodspiv[f'{group}_median'] = femaleadultmodspiv[cols].median(axis=1)

femaleadultmodspiv.head()

,Prot,50aa_reg_range,dNQ_JA2_F (ms-status_RAT8),dNQ_JA5_F (ms-status_RAT11),dNQ_JA6_F (ms-status_RAT12),oxK_JA2_F (ms-status_RAT8),oxK_JA5_F (ms-status_RAT11),oxK_JA6_F (ms-status_RAT12),oxM_JA2_F (ms-status_RAT8),oxM_JA5_F (ms-status_RAT11),oxM_JA6_F (ms-status_RAT12),oxP_JA2_F (ms-status_RAT8),oxP_JA5_F (ms-status_RAT11),oxP_JA6_F (ms-status_RAT12),AVG. dNQ_median,AVG. oxK_median,AVG. oxM_median,AVG. oxP_median
0,Apoe,0 - 50,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,Apoe,100 - 150,2,5,2,0,0,0,4,6,5,0,0,0,2.0,0.0,5.0,0.0
2,Apoe,150 - 200,0,1,1,0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
3,Apoe,200 - 250,1,2,0,0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
4,Apoe,250 - 300,2,3,1,2,2,2,3,7,6,2,2,2,2.0,2.0,6.0,2.0


In [17]:
dNQCols = ['dNQ_FZ1_F (ms-status_RAT1)', 'dNQ_FZ2_F (ms-status_RAT2)', 'dNQ_FZ6_M (ms-status_RAT6)']
oxKCols = ['oxK_FZ1_F (ms-status_RAT1)', 'oxK_FZ2_F (ms-status_RAT2)', 'oxK_FZ6_M (ms-status_RAT6)']
oxMCols = ['oxM_FZ1_F (ms-status_RAT1)', 'oxM_FZ2_F (ms-status_RAT2)', 'oxM_FZ6_M (ms-status_RAT6)']
oxPCols = ['oxP_FZ1_F (ms-status_RAT1)', 'oxP_FZ2_F (ms-status_RAT2)', 'oxP_FZ6_M (ms-status_RAT6)']

for group, cols in {
    'AVG. dNQ': dNQCols,
    'AVG. oxK': oxKCols,
    'AVG. oxM': oxMCols,
    'AVG. oxP': oxPCols
}.items():
    femalejuvmodspiv[f'{group}_median'] = femalejuvmodspiv[cols].median(axis=1)

femalejuvmodspiv.head()


,Prot,50aa_reg_range,dNQ_FZ1_F (ms-status_RAT1),dNQ_FZ2_F (ms-status_RAT2),dNQ_FZ6_M (ms-status_RAT6),oxK_FZ1_F (ms-status_RAT1),oxK_FZ2_F (ms-status_RAT2),oxK_FZ6_M (ms-status_RAT6),oxM_FZ1_F (ms-status_RAT1),oxM_FZ2_F (ms-status_RAT2),oxM_FZ6_M (ms-status_RAT6),oxP_FZ1_F (ms-status_RAT1),oxP_FZ2_F (ms-status_RAT2),oxP_FZ6_M (ms-status_RAT6),AVG. dNQ_median,AVG. oxK_median,AVG. oxM_median,AVG. oxP_median
0,Apoe,0 - 50,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,Apoe,100 - 150,0,2,1,0,0,0,3,6,5,0,0,0,1.0,0.0,5.0,0.0
2,Apoe,150 - 200,0,1,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
3,Apoe,200 - 250,0,3,2,0,0,0,0,0,0,0,0,0,2.0,0.0,0.0,0.0
4,Apoe,250 - 300,0,2,1,0,0,0,1,4,3,0,0,0,1.0,0.0,3.0,0.0


In [18]:
dNQCols = ['dNQ_FZ3_F (ms-status_RAT3)', 'dNQ_FZ4_M (ms-status_RAT4)', 'dNQ_FZ5_M (ms-status_RAT5)']
oxKCols = ['oxK_FZ3_F (ms-status_RAT3)', 'oxK_FZ4_M (ms-status_RAT4)', 'oxK_FZ5_M (ms-status_RAT5)']
oxMCols = ['oxM_FZ3_F (ms-status_RAT3)', 'oxM_FZ5_M (ms-status_RAT5)', 'oxM_FZ5_M (ms-status_RAT5)']
oxPCols = ['oxP_FZ3_F (ms-status_RAT3)', 'oxP_FZ4_M (ms-status_RAT4)', 'oxP_FZ5_M (ms-status_RAT5)']

for group, cols in {
    'AVG. dNQ': dNQCols,
    'AVG. oxK': oxKCols,
    'AVG. oxM': oxMCols,
    'AVG. oxP': oxPCols
}.items():
    malejuvmodspiv[f'{group}_median'] = malejuvmodspiv[cols].median(axis=1)

malejuvmodspiv.head()


,Prot,50aa_reg_range,dNQ_FZ3_F (ms-status_RAT3),dNQ_FZ4_M (ms-status_RAT4),dNQ_FZ5_M (ms-status_RAT5),oxK_FZ3_F (ms-status_RAT3),oxK_FZ4_M (ms-status_RAT4),oxK_FZ5_M (ms-status_RAT5),oxM_FZ3_F (ms-status_RAT3),oxM_FZ4_M (ms-status_RAT4),oxM_FZ5_M (ms-status_RAT5),oxP_FZ3_F (ms-status_RAT3),oxP_FZ4_M (ms-status_RAT4),oxP_FZ5_M (ms-status_RAT5),AVG. dNQ_median,AVG. oxK_median,AVG. oxM_median,AVG. oxP_median
0,Apoe,0 - 50,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,Apoe,100 - 150,0,1,0,0,0,0,2,4,2,0,0,0,0.0,0.0,2.0,0.0
2,Apoe,150 - 200,1,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
3,Apoe,200 - 250,0,2,2,0,0,0,0,0,0,0,0,0,2.0,0.0,0.0,0.0
4,Apoe,250 - 300,0,0,1,0,0,0,2,2,1,0,0,0,0.0,0.0,1.0,0.0


In [19]:
dNQCols = ['dNQ_JA1_M (ms-status_RAT7)', 'dNQ_JA3_M (ms-status_RAT9)', 'dNQ_JA4_M (ms-status_RAT10)']
oxKCols = ['oxK_JA1_M (ms-status_RAT7)', 'oxK_JA3_M (ms-status_RAT9)', 'oxK_JA4_M (ms-status_RAT10)']
oxMCols = ['oxM_JA1_M (ms-status_RAT7)', 'oxM_JA3_M (ms-status_RAT9)', 'oxM_JA4_M (ms-status_RAT10)']
oxPCols = ['oxP_JA1_M (ms-status_RAT7)', 'oxP_JA3_M (ms-status_RAT9)', 'oxP_JA4_M (ms-status_RAT10)']

for group, cols in {
    'AVG. dNQ': dNQCols,
    'AVG. oxK': oxKCols,
    'AVG. oxM': oxMCols,
    'AVG. oxP': oxPCols
}.items():
    maleadultmodspiv[f'{group}_median'] = maleadultmodspiv[cols].median(axis=1)

maleadultmodspiv.head()


,Prot,50aa_reg_range,dNQ_JA1_M (ms-status_RAT7),dNQ_JA3_M (ms-status_RAT9),dNQ_JA4_M (ms-status_RAT10),oxK_JA1_M (ms-status_RAT7),oxK_JA3_M (ms-status_RAT9),oxK_JA4_M (ms-status_RAT10),oxM_JA1_M (ms-status_RAT7),oxM_JA3_M (ms-status_RAT9),oxM_JA4_M (ms-status_RAT10),oxP_JA1_M (ms-status_RAT7),oxP_JA3_M (ms-status_RAT9),oxP_JA4_M (ms-status_RAT10),AVG. dNQ_median,AVG. oxK_median,AVG. oxM_median,AVG. oxP_median
0,Apoe,0 - 50,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,Apoe,100 - 150,2,2,5,0,0,0,3,4,6,0,0,0,2.0,0.0,4.0,0.0
2,Apoe,150 - 200,1,0,1,0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
3,Apoe,200 - 250,1,1,2,0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
4,Apoe,250 - 300,3,2,3,2,2,2,4,3,7,2,2,2,3.0,2.0,4.0,2.0


In [50]:
maleadultmodspiv

,Prot,50aa_reg_range,dNQ_JA1_M (ms-status_RAT7),dNQ_JA3_M (ms-status_RAT9),dNQ_JA4_M (ms-status_RAT10),oxK_JA1_M (ms-status_RAT7),oxK_JA3_M (ms-status_RAT9),oxK_JA4_M (ms-status_RAT10),oxM_JA1_M (ms-status_RAT7),oxM_JA3_M (ms-status_RAT9),oxM_JA4_M (ms-status_RAT10),oxP_JA1_M (ms-status_RAT7),oxP_JA3_M (ms-status_RAT9),oxP_JA4_M (ms-status_RAT10),AVG. dNQ_median,AVG. oxK_median,AVG. oxM_median,AVG. oxP_median
0,Apoe,0 - 50,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
1,Apoe,100 - 150,2,2,5,0,0,0,3,4,6,0,0,0,2.0,0.0,4.0,0.0
2,Apoe,150 - 200,1,0,1,0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
3,Apoe,200 - 250,1,1,2,0,0,0,0,0,0,0,0,0,1.0,0.0,0.0,0.0
4,Apoe,250 - 300,3,2,3,2,2,2,4,3,7,2,2,2,3.0,2.0,4.0,2.0
5,Apoe,50 - 100,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
6,Col1a1,0 - 50,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
7,Col1a1,100 - 150,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0
8,Col1a1,1000 - 1050,0,0,1,1,4,3,15,18,16,70,103,90,0.0,3.0,16.0,90.0
9,Col1a1,1050 - 1100,3,2,3,1,1,0,0,0,0,23,39,32,3.0,1.0,0.0,32.0


In [21]:
fjgids = ['FZ1_F (ms-status_RAT1)', 'FZ2_F (ms-status_RAT2)', 'FZ3_F (ms-status_RAT3)']
mjgids = ['FZ4_M (ms-status_RAT4)', 'FZ5_M (ms-status_RAT5)', 'FZ6_M (ms-status_RAT6)']

fagids = ['JA2_F (ms-status_RAT8)', 'JA5_F (ms-status_RAT11)', 'JA6_F (ms-status_RAT12)']
magids = ['JA1_M (ms-status_RAT7)', 'JA3_M (ms-status_RAT9)', 'JA4_M (ms-status_RAT10)']

In [22]:
maleptms = df[(df['sample'] == 'FZ4_M (ms-status_RAT4)') | (df['sample'] == 'FZ5_M (ms-status_RAT5)') | (df['sample'] == 'FZ6_M (ms-status_RAT6)') | (df['sample'] == 'JA1_M (ms-status_RAT7)') | (df['sample'] == 'JA3_M (ms-status_RAT9)') | (df['sample'] == 'JA4_M (ms-status_RAT10)')]
femaleptms = df[(df['sample'] == 'FZ1_F (ms-status_RAT1)') | (df['sample'] == 'FZ2_F (ms-status_RAT2)') | (df['sample'] == 'FZ3_F (ms-status_RAT3)') |(df['sample'] == 'JA2_F (ms-status_RAT8)') | (df['sample'] == 'JA5_F (ms-status_RAT11)') | (df['sample'] == 'JA6_F (ms-status_RAT12)')]

In [23]:
maleptms = findtotalptms([maleptms], proteinList)
femaleptms = findtotalptms([femaleptms], proteinList)

ldslabels ['FZ4_M (ms-status_RAT4)' 'FZ5_M (ms-status_RAT5)'
 'FZ6_M (ms-status_RAT6)' 'JA1_M (ms-status_RAT7)'
 'JA3_M (ms-status_RAT9)' 'JA4_M (ms-status_RAT10)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


ldslabels ['FZ1_F (ms-status_RAT1)' 'FZ2_F (ms-status_RAT2)'
 'FZ3_F (ms-status_RAT3)' 'JA2_F (ms-status_RAT8)'
 'JA5_F (ms-status_RAT11)' 'JA6_F (ms-status_RAT12)']


C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].fillna("")
C:\Users\jtudo\AppData\Local\Temp\ipykernel_23936\3987044331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds["pep_mods"] = ds["pep_mods"].astype(str)


In [24]:
pivmaleptms = pivotmods(maleptms)
pivfemaleptms = pivotmods(femaleptms)

In [25]:
# Testing Normalisation

In [26]:
df_female = pd.read_csv('../data/PLF_Buckley_Female_YFZvsOJA.tsv', sep='\t')
df_male = pd.read_csv('../data/PLF_Buckley_Male_YFZvsOJA.tsv', sep='\t')

In [29]:
# prot = df[df['Uniprot_ID'] == 'CO1A1_RAT']
# prot[prot['Uniprot_ID'] == 'CO1A1_RAT']

In [33]:
def retrieveScaleFactors(dataset, agegroup):
    data = dataset.filter(like=agegroup)
    
    mediantotalcount = data.filter(like="Original").sum(axis=0).median()
    #all_values = data.filter(like="Original").values  # array of all segment counts
    #max_count = all_values.max()
    #min_count = all_values.min()
    
    # print("Max count:", max_count)
    # print("Min count:", min_count)
    # mediantotalcount = statistics.median([min_count, max_count])
    # print(mediantotalcount)
    
    sumtotal = data.filter(like="Original").sum(axis=0)
    # print(sumtotal)
    
    ScalingFactors = mediantotalcount / sumtotal
    # print(type(ScalingFactors), ScalingFactors)
    
    
    cols_to_normalize = data.filter(like="Original").columns

    factors = {}
    
    # Apply scaling factors column by column
    for i, col in enumerate(cols_to_normalize):
        key = f'{col}_NORM'
        val = ScalingFactors[col]
        factors[key] = val 
        # print(i, col, ScalingFactors[col])
        # data[f'{col}_median'] = data[col] * ScalingFactors[col]
        #data[f'{col}_median'] = np.floor(data[f'{col}_median'] + 0.5).astype(int)
    #Old
    return factors

In [34]:
femaleNFs = {}
for p in ['APOE_RAT', 'CO1A1_RAT', 'CO1A2_RAT', 'CO2A1_RAT', 'COMP_RAT', 'OSTP_RAT', 'THRB_RAT']:
    if p in df_female['Uniprot_ID'].unique():
        data = df_female[df_female['Uniprot_ID']== p]
        key = f'{p}_FZ'
        femaleNFs[key] = FuzzyFemaleSF = retrieveScaleFactors(data, 'FZ')
        key = f'{p}_JA'
        femaleNFs[key] = JumboFemaleSF = retrieveScaleFactors(data, 'JA')

In [36]:
maleNFs = {}
for p in ['APOE_RAT', 'CO1A1_RAT', 'CO1A2_RAT', 'CO2A1_RAT', 'COMP_RAT', 'OSTP_RAT', 'THRB_RAT']:
    if p in df_male['Uniprot_ID'].unique():
        data = df_male[df_male['Uniprot_ID']== p]
        key = f'{p}_FZ'
        maleNFs[key] = FuzzyMaleSF = retrieveScaleFactors(data, 'FZ')
        key = f'{p}_JA'
        maleNFs[key] = JumboMaleSF = retrieveScaleFactors(data, 'JA')

In [38]:
import re
malescalingfactorsdf = pivmaleptms.copy()

for key, value in maleNFs.items():
    protein = key.split('_')[0]
    if protein == 'OSTP':
        protein = 'Spp1'
    elif protein == 'CO1A1':
        protein = 'Col1a1'
    elif protein == 'CO1A2':
        protein = 'Col1a2'
    elif protein == 'CO2A1':
        protein = 'Col2a1'
    elif protein == 'THRB':
        protien = 'F2'
    else:
        protein = protein
    for Key, Value in value.items():
        print(f'key: {Key}', f'value: {Value}')

        pattern = r'(FZ\d+|JA\d+)'
        
        match = re.search(pattern, str(Key))
        if match:
            code = match.group(0)
            print(f"Extracted code: {code}")

        mask_rows = malescalingfactorsdf['Prot'].str.contains(protein, case=False)

        # Columns whose name contains 'math'
        mask_cols = [col for col in malescalingfactorsdf.columns if code in col]

        print(f'updating criteria {protein} {code}')
        
        # Update those cells,
        malescalingfactorsdf.loc[mask_rows, mask_cols] = float(Value)


key: Original_FZ4_M (ms-status_RAT4)_NORM value: 1.0
Extracted code: FZ4
updating criteria APOE FZ4
key: Original_FZ5_M (ms-status_RAT5)_NORM value: 1.2105263157894737
Extracted code: FZ5
updating criteria APOE FZ5
key: Original_FZ6_M (ms-status_RAT6)_NORM value: 1.0
Extracted code: FZ6
updating criteria APOE FZ6
key: Original_JA1_M (ms-status_RAT7)_NORM value: 1.0
Extracted code: JA1
updating criteria APOE JA1
key: Original_JA3_M (ms-status_RAT9)_NORM value: 1.2413793103448276
Extracted code: JA3
updating criteria APOE JA3
key: Original_JA4_M (ms-status_RAT10)_NORM value: 0.972972972972973
Extracted code: JA4
updating criteria APOE JA4
key: Original_FZ4_M (ms-status_RAT4)_NORM value: 1.0
Extracted code: FZ4
updating criteria Col1a2 FZ4
key: Original_FZ5_M (ms-status_RAT5)_NORM value: 1.19375
Extracted code: FZ5
updating criteria Col1a2 FZ5
key: Original_FZ6_M (ms-status_RAT6)_NORM value: 0.9408866995073891
Extracted code: FZ6
updating criteria Col1a2 FZ6
key: Original_JA1_M (ms-status

In [40]:
femalescalingfactorsdf = pivfemaleptms.copy()

for key, value in femaleNFs.items():
    protein = key.split('_')[0]
    if protein == 'OSTP':
        protein = 'Spp1'
    elif protein == 'CO1A1':
        protein = 'Col1a1'
    elif protein == 'CO1A2':
        protein = 'Col1a2'
    elif protein == 'CO2A1':
        protein = 'Col2a1'
    elif protein == 'THRB':
        protien = 'F2'
    else:
        protein = protein
    for Key, Value in value.items():
        print(f'key: {Key}', f'value: {Value}')

        pattern = r'(FZ\d+|JA\d+)'
        
        match = re.search(pattern, str(Key))
        if match:
            code = match.group(0)
            print(f"Extracted code: {code}")

        mask_rows = femalescalingfactorsdf['Prot'].str.contains(protein, case=False)

        # Columns whose name contains 'math'
        mask_cols = [col for col in femalescalingfactorsdf.columns if code in col]

        print(f'updating criteria {protein} {code}')
        
        # Update those cells,
        femalescalingfactorsdf.loc[mask_rows, mask_cols] = float(Value)


key: Original_FZ1_F (ms-status_RAT1)_NORM value: 1.0769230769230769
Extracted code: FZ1
updating criteria APOE FZ1
key: Original_FZ2_F (ms-status_RAT2)_NORM value: 0.45161290322580644
Extracted code: FZ2
updating criteria APOE FZ2
key: Original_FZ3_F (ms-status_RAT3)_NORM value: 1.0
Extracted code: FZ3
updating criteria APOE FZ3
key: Original_JA2_F (ms-status_RAT8)_NORM value: 1.0344827586206897
Extracted code: JA2
updating criteria APOE JA2
key: Original_JA5_F (ms-status_RAT11)_NORM value: 0.8108108108108109
Extracted code: JA5
updating criteria APOE JA5
key: Original_JA6_F (ms-status_RAT12)_NORM value: 1.0
Extracted code: JA6
updating criteria APOE JA6
key: Original_FZ1_F (ms-status_RAT1)_NORM value: 1.008298755186722
Extracted code: FZ1
updating criteria Col1a1 FZ1
key: Original_FZ2_F (ms-status_RAT2)_NORM value: 0.9310344827586207
Extracted code: FZ2
updating criteria Col1a1 FZ2
key: Original_FZ3_F (ms-status_RAT3)_NORM value: 1.0
Extracted code: FZ3
updating criteria Col1a1 FZ3
ke

In [42]:
normalisedmaledf = pivmaleptms.copy()
normalisedfemaledf = pivfemaleptms.copy()

malenum_cols = normalisedmaledf.select_dtypes(include='float').columns
femalenum_cols = normalisedfemaledf.select_dtypes(include='float').columns

normalisedmaledf[malenum_cols] = np.round(normalisedmaledf[malenum_cols] * malescalingfactorsdf[malenum_cols])
normalisedfemaledf[femalenum_cols] = np.round(normalisedfemaledf[femalenum_cols] * femalescalingfactorsdf[femalenum_cols])

In [44]:
RegionsToSearch = {
    "col1a1": [
        (950, 1000),
        (1000, 1050)
    ],
    "col1a2": [
        (1000, 1050),
        (1050, 1100)
    ],
    "col2a1": [
        (550, 600),
        (600, 650),
        (800, 850),
        (850, 900),
        (1150, 1200),
        (1200, 1250),
        (1300, 1350),
        (1350, 1400)
    ],
    "spp1": [
        (0, 50),
        (150, 200)
    ],
    "F2": [
        (350, 400)
    ],
    "APOE": [
        (200, 250),
        (250, 300)
    ],
    "Comp": [
        (600, 650)
    ]
}

In [49]:
teststring = '400 - 450'
teststring.split()

['400', '-', '450']

In [47]:
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# Example: load your dataframe
# df = pd.read_csv("your_data.csv")  
def performPTMAnalysis:

    # Define RegionsToSearch
    RegionsToSearch = {
        "col1a1": [(950, 1000), (1000, 1050)],
        "col1a2": [(1000, 1050), (1050, 1100)],
        "col2a1": [(550, 600), (600, 650), (800, 850), (850, 900), (1150, 1200), (1200, 1250), (1300, 1350), (1350, 1400)],
        "spp1": [(0, 50), (150, 200)],
        "F2": [(350, 400)],
        "APOE": [(200, 250), (250, 300)],
        "Comp": [(600, 650)]
    }

    # Modifications and corresponding columns
    mods = ["dNQ", "oxK", "oxP", "oxM"]

    # Function to parse string range like "950-1000" to tuple (950, 1000)
    for key, value in RegionsToSearch:
        protein = key
        regions = value

        for r in regions:
            low = r[0]
            high = r[1]
            
            for mod in mods:
                relcols = data.columns.contains(mod)
                
                data = d[(d['Prot'] == protein) and (d['50aa_reg_range'].split()[0] == low) and (d['50aa_reg_range'].split()[1] == high)]
                
                
